In [1]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import requests
import gzip
from collections import defaultdict, Counter
from typing import Dict, List, Tuple, Any, Optional
import warnings
warnings.filterwarnings('ignore')

# Configuração para visualização
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

# Configuração de reproducibilidade
np.random.seed(42)

In [2]:
class TemporalNetworkAnalyzer:
    """
    Analisador principal para redes temporais do dataset CollegeMsg.
    
    Atributos:
        edges (pd.DataFrame): DataFrame com as arestas temporais
        nodes (set): Conjunto de todos os nós na rede
        temporal_graphs (dict): Dicionário com grafos NetworkX para cada período
        time_bins (pd.DatetimeIndex): Bins temporais para agregação
        time_window (str): Janela temporal utilizada ('H', 'D', 'W')
    """
    
    def __init__(self):
        """Inicializa o analisador de redes temporais."""
        self.edges = None
        self.nodes = None
        self.temporal_graphs = {}
        self.time_bins = None
        self.time_window = 'D'  # Default: diário
        
    def download_dataset(self, url: str = "https://snap.stanford.edu/data/CollegeMsg.txt.gz") -> bool:
        """
        Baixa o dataset CollegeMsg da Stanford SNAP.
        
        Args:
            url (str): URL do dataset
            
        Returns:
            bool: True se o download foi bem-sucedido
        """
        try:
            print("Baixando dataset CollegeMsg...")
            response = requests.get(url, stream=True)
            response.raise_for_status()
            
            filename = "CollegeMsg.txt.gz"
            with open(filename, 'wb') as f:
                for chunk in response.iter_content(chunk_size=8192):
                    f.write(chunk)
            
            print(f"✓ Dataset baixado: {filename}")
            return True
            
        except Exception as e:
            print(f"✗ Erro no download: {e}")
            return False
    
    def load_dataset(self, filepath: str = "CollegeMsg.txt.gz") -> bool:
        """
        Carrega o dataset CollegeMsg em memória.
        
        Args:
            filepath (str): Caminho para o arquivo do dataset
            
        Returns:
            bool: True se o carregamento foi bem-sucedido
        """
        try:
            print("Carregando dataset...")
            
            # Lê arquivo comprimido ou não comprimido
            if filepath.endswith('.gz'):
                with gzip.open(filepath, 'rt') as f:
                    data = []
                    for line in f:
                        if not line.startswith('#'):
                            parts = line.strip().split()
                            if len(parts) == 3:
                                data.append([int(parts[0]), int(parts[1]), int(parts[2])])
            else:
                data = []
                with open(filepath, 'r') as f:
                    for line in f:
                        if not line.startswith('#'):
                            parts = line.strip().split()
                            if len(parts) == 3:
                                data.append([int(parts[0]), int(parts[1]), int(parts[2])])
            
            # Cria DataFrame
            self.edges = pd.DataFrame(data, columns=['source', 'target', 'timestamp'])
            
            # Converte timestamp para datetime
            self.edges['datetime'] = pd.to_datetime(self.edges['timestamp'], unit='s')
            
            # Identifica todos os nós únicos
            self.nodes = set(self.edges['source'].unique()) | set(self.edges['target'].unique())
            
            print(f"✓ Dataset carregado:")
            print(f"  - {len(self.edges)} arestas temporais")
            print(f"  - {len(self.nodes)} nós únicos")
            print(f"  - Período: {self.edges['datetime'].min()} a {self.edges['datetime'].max()}")
            
            return True
            
        except Exception as e:
            print(f"✗ Erro no carregamento: {e}")
            return False
    
    def preprocess_data(self, time_window: str = 'D') -> None:
        """
        Preprocessa os dados criando bins temporais.
        
        Args:
            time_window (str): Janela temporal ('H'=horário, 'D'=diário, 'W'=semanal)
        """
        if self.edges is None:
            raise ValueError("Dataset não carregado. Execute load_dataset() primeiro.")
        
        self.time_window = time_window
        
        print(f"Preprocessando dados com janela temporal: {time_window}")
        
        # Cria bins temporais
        start_time = self.edges['datetime'].min()
        end_time = self.edges['datetime'].max()
        
        self.time_bins = pd.date_range(start=start_time, end=end_time, freq=time_window)
        
        # Atribui cada aresta a um bin temporal
        self.edges['time_bin'] = pd.cut(self.edges['datetime'], bins=self.time_bins, 
                                       labels=False, include_lowest=True)
        
        # Remove arestas que não se encaixam em nenhum bin
        self.edges = self.edges.dropna(subset=['time_bin'])
        self.edges['time_bin'] = self.edges['time_bin'].astype(int)
        
        print(f"✓ Dados preprocessados:")
        print(f"  - {len(self.time_bins)-1} períodos temporais")
        print(f"  - {len(self.edges)} arestas válidas")
    
    def create_temporal_snapshots(self) -> None:
        """
        Cria snapshots da rede para cada período temporal.
        """
        if self.edges is None or self.time_bins is None:
            raise ValueError("Execute preprocess_data() primeiro.")
        
        print("Criando snapshots temporais...")
        
        self.temporal_graphs = {}
        
        for bin_idx in range(len(self.time_bins) - 1):
            # Filtra arestas do período atual
            period_edges = self.edges[self.edges['time_bin'] == bin_idx]
            
            # Cria grafo para o período
            G = nx.Graph()
            
            # Adiciona nós (todos os nós que existem no dataset)
            G.add_nodes_from(self.nodes)
            
            # Adiciona arestas do período
            if not period_edges.empty:
                edge_list = list(zip(period_edges['source'], period_edges['target']))
                G.add_edges_from(edge_list)
            
            self.temporal_graphs[bin_idx] = G
        
        print(f"✓ {len(self.temporal_graphs)} snapshots criados")
        
        # Estatísticas básicas
        avg_nodes = np.mean([G.number_of_nodes() for G in self.temporal_graphs.values()])
        avg_edges = np.mean([G.number_of_edges() for G in self.temporal_graphs.values()])
        
        print(f"  - Média de nós por snapshot: {avg_nodes:.1f}")
        print(f"  - Média de arestas por snapshot: {avg_edges:.1f}")


In [3]:
class HubAnalyzer:
    """Analisador corrigido para identificação e rastreamento de hubs temporais."""
    
    def __init__(self, analyzer):
        self.analyzer = analyzer
        self.centrality_history = {}
        self.hub_history = {}
        self.hub_persistence = {}
        self.persistent_hubs = []
        self.method_used = 'degree'
    
    def calculate_centrality_metrics(self, graph):
        """Calcula múltiplas métricas de centralidade para um grafo."""
        centralities = {}
        centralities['degree'] = nx.degree_centrality(graph)
        
        if graph.number_of_nodes() > 1 and graph.number_of_edges() > 0:
            try:
                centralities['betweenness'] = nx.betweenness_centrality(graph)
            except:
                centralities['betweenness'] = {node: 0.0 for node in graph.nodes()}
            
            try:
                if nx.is_connected(graph):
                    centralities['closeness'] = nx.closeness_centrality(graph)
                else:
                    largest_cc = max(nx.connected_components(graph), key=len)
                    subgraph = graph.subgraph(largest_cc)
                    cc_closeness = nx.closeness_centrality(subgraph)
                    centralities['closeness'] = {node: cc_closeness.get(node, 0.0) 
                                               for node in graph.nodes()}
            except:
                centralities['closeness'] = {node: 0.0 for node in graph.nodes()}
            
            try:
                centralities['eigenvector'] = nx.eigenvector_centrality(graph, max_iter=1000)
            except:
                centralities['eigenvector'] = {node: 0.0 for node in graph.nodes()}
        else:
            centralities['betweenness'] = {node: 0.0 for node in graph.nodes()}
            centralities['closeness'] = {node: 0.0 for node in graph.nodes()}
            centralities['eigenvector'] = {node: 0.0 for node in graph.nodes()}
        
        return centralities
    
    def identify_hubs(self, graph, top_k=10, method='degree'):
        """Identifica os top-k hubs em um grafo."""
        centralities = self.calculate_centrality_metrics(graph)
        if method not in centralities:
            method = 'degree'
        sorted_nodes = sorted(centralities[method].items(), key=lambda x: x[1], reverse=True)
        return sorted_nodes[:top_k]
    
    def analyze_temporal_hubs(self, top_k=10, method='degree'):
        """Analisa a evolução temporal dos hubs."""
        if not self.analyzer.temporal_graphs:
            raise ValueError("Snapshots temporais não criados.")
        
        print(f"Analisando evolução temporal dos hubs (método: {method}, top-{top_k})...")
        
        self.centrality_history = {}
        self.hub_history = {}
        
        for period, graph in self.analyzer.temporal_graphs.items():
            centralities = self.calculate_centrality_metrics(graph)
            self.centrality_history[period] = centralities
            
            hubs = self.identify_hubs(graph, top_k, method)
            self.hub_history[period] = hubs
        
        all_hubs = set()
        for hubs in self.hub_history.values():
            all_hubs.update([hub[0] for hub in hubs])
        
        hub_persistence = {}
        for hub in all_hubs:
            appearances = sum(1 for hubs in self.hub_history.values() 
                            if hub in [h[0] for h in hubs])
            hub_persistence[hub] = appearances / len(self.hub_history)
        
        persistent_hubs = sorted(hub_persistence.items(), key=lambda x: x[1], reverse=True)
        
        # CORREÇÃO: Salvar como atributos da classe
        self.hub_persistence = hub_persistence
        self.persistent_hubs = persistent_hubs
        self.method_used = method
        
        print(f"✓ Análise concluída:")
        print(f"  - {len(all_hubs)} nós identificados como hubs")
        if persistent_hubs:
            print(f"  - Top 3 hubs mais persistentes:")
            for i, (hub, persistence) in enumerate(persistent_hubs[:3]):
                print(f"    {i+1}. Nó {hub}: {persistence:.1%} do tempo")
        
        return {
            'centrality_history': self.centrality_history,
            'hub_history': self.hub_history,
            'hub_persistence': hub_persistence,
            'persistent_hubs': persistent_hubs,
            'method_used': method,
            'top_k': top_k
        }
    
    def hub_statistics(self):
        """Calcula estatísticas detalhadas sobre os hubs."""
        if not self.hub_history:
            raise ValueError("Execute analyze_temporal_hubs() primeiro.")
        
        stats = {}
        all_hubs = set()
        for hubs in self.hub_history.values():
            all_hubs.update([hub[0] for hub in hubs])
        stats['total_unique_hubs'] = len(all_hubs)
        
        hub_appearances = Counter()
        for hubs in self.hub_history.values():
            for hub, _ in hubs:
                hub_appearances[hub] += 1
        
        total_periods = len(self.hub_history)
        
        highly_persistent = [hub for hub, count in hub_appearances.items() 
                           if count / total_periods >= 0.8]
        moderately_persistent = [hub for hub, count in hub_appearances.items() 
                               if 0.5 <= count / total_periods < 0.8]
        low_persistent = [hub for hub, count in hub_appearances.items() 
                        if count / total_periods < 0.5]
        
        stats['highly_persistent_hubs'] = len(highly_persistent)
        stats['moderately_persistent_hubs'] = len(moderately_persistent)
        stats['low_persistent_hubs'] = len(low_persistent)
        
        return stats

In [4]:
class TemporalClusterAnalyzer:
    """
    Analisador especializado para detecção e análise temporal de comunidades.
    
    Atributos:
        analyzer (TemporalNetworkAnalyzer): Referência ao analisador principal
        community_history (dict): Histórico de comunidades por período
        modularity_history (dict): Histórico de modularidade por período
    """
    
    def __init__(self, analyzer: TemporalNetworkAnalyzer):
        """
        Inicializa o analisador de comunidades.
        
        Args:
            analyzer (TemporalNetworkAnalyzer): Instância do analisador principal
        """
        self.analyzer = analyzer
        self.community_history = {}
        self.modularity_history = {}
        self.num_communities_per_period = []  # Atributo inicializado
        self.avg_community_size_per_period = []  # Atributo inicializado
        self.stability_scores = []
        self.avg_stability = 0.0
    
    def detect_communities(self, graph: nx.Graph, method: str = 'louvain') -> Tuple[Dict, float]:
        """
        Detecta comunidades em um grafo usando o método especificado.
        
        Args:
            graph (nx.Graph): Grafo para detecção de comunidades
            method (str): Método de detecção ('louvain', 'greedy')
            
        Returns:
            Tuple[Dict, float]: Dicionário de comunidades e modularidade
        """
        if graph.number_of_edges() == 0:
            # Grafo sem arestas - cada nó é sua própria comunidade
            communities = {node: i for i, node in enumerate(graph.nodes())}
            modularity = 0.0
            return communities, modularity
        
        try:
            if method == 'louvain':
                # Implementação manual do algoritmo de Louvain simplificado
                # Como não temos python-louvain, usamos greedy modularity communities
                communities_sets = nx.algorithms.community.greedy_modularity_communities(graph)
                
                # Converte para dicionário node -> community_id
                communities = {}
                for i, community_set in enumerate(communities_sets):
                    for node in community_set:
                        communities[node] = i
                
                # Calcula modularidade
                modularity = nx.algorithms.community.modularity(graph, communities_sets)
                
            else:  # método greedy padrão
                communities_sets = nx.algorithms.community.greedy_modularity_communities(graph)
                
                communities = {}
                for i, community_set in enumerate(communities_sets):
                    for node in community_set:
                        communities[node] = i
                
                modularity = nx.algorithms.community.modularity(graph, communities_sets)
            
        except Exception as e:
            print(f"Erro na detecção de comunidades: {e}")
            # Fallback: cada nó é sua própria comunidade
            communities = {node: i for i, node in enumerate(graph.nodes())}
            modularity = 0.0
        
        return communities, modularity
    
    def analyze_temporal_communities(self, method='louvain'):
            if not self.analyzer.temporal_graphs:
                raise ValueError("Snapshots temporais não criados.")
            
            print(f"Analisando evolução temporal das comunidades (método: {method})...")
            
            # Resetar listas a cada nova análise
            self.num_communities_per_period = []
            self.avg_community_size_per_period = []
            self.stability_scores = []
            
            for period, graph in self.analyzer.temporal_graphs.items():
                communities, modularity = self.detect_communities(graph, method)
                self.community_history[period] = communities
                self.modularity_history[period] = modularity
                
                unique_communities = set(communities.values())
                num_communities = len(unique_communities)
                self.num_communities_per_period.append(num_communities)
                
                if num_communities > 0:
                    community_sizes = Counter(communities.values())
                    avg_size = np.mean(list(community_sizes.values()))
                    self.avg_community_size_per_period.append(avg_size)
                else:
                    self.avg_community_size_per_period.append(0)
            
            # Cálculo de estabilidade entre períodos consecutivos
            periods = sorted(self.community_history.keys())
            for i in range(len(periods) - 1):
                stability = self._calculate_community_stability(
                    self.community_history[periods[i]],
                    self.community_history[periods[i+1]]
                )
                self.stability_scores.append(stability)
            
            self.avg_stability = np.mean(self.stability_scores) if self.stability_scores else 0.0
            
            print(f"✓ Análise concluída:")
            print(f"  - Média de comunidades: {np.mean(self.num_communities_per_period):.1f}")
            print(f"  - Tamanho médio: {np.mean(self.avg_community_size_per_period):.1f}")
            print(f"  - Estabilidade: {self.avg_stability:.3f}")
            print(f"  - Modularidade: {np.mean(list(self.modularity_history.values())):.3f}")
            
            return {
                'community_history': self.community_history,
                'modularity_history': self.modularity_history,
                'num_communities_per_period': self.num_communities_per_period,
                'avg_community_size_per_period': self.avg_community_size_per_period,
                'stability_scores': self.stability_scores,
                'avg_stability': self.avg_stability,
                'method_used': method
            }
    
    def _calculate_community_stability(self, communities1: Dict, communities2: Dict) -> float:
        """
        Calcula a estabilidade entre duas detecções de comunidades usando Adjusted Rand Index.
        
        Args:
            communities1 (Dict): Comunidades do primeiro período
            communities2 (Dict): Comunidades do segundo período
            
        Returns:
            float: Índice de estabilidade (0-1)
        """
        # Encontra nós comuns entre os dois períodos
        common_nodes = set(communities1.keys()) & set(communities2.keys())
        
        if len(common_nodes) < 2:
            return 0.0
        
        # Cria listas de labels para os nós comuns
        labels1 = [communities1[node] for node in common_nodes]
        labels2 = [communities2[node] for node in common_nodes]
        
        # Calcula Adjusted Rand Index manualmente
        return self._adjusted_rand_index(labels1, labels2)
    
    def _adjusted_rand_index(self, labels1: List, labels2: List) -> float:
        """
        Calcula o Adjusted Rand Index entre duas partições.
        
        Args:
            labels1 (List): Labels da primeira partição
            labels2 (List): Labels da segunda partição
            
        Returns:
            float: Adjusted Rand Index
        """
        n = len(labels1)
        if n == 0:
            return 0.0
        
        # Cria tabela de contingência
        contingency_table = defaultdict(lambda: defaultdict(int))
        for l1, l2 in zip(labels1, labels2):
            contingency_table[l1][l2] += 1
        
        # Calcula somas marginais
        sum_comb_c = sum(self._comb2(sum(row.values())) for row in contingency_table.values())
        sum_comb_k = sum(self._comb2(sum(contingency_table[i][j] for i in contingency_table))
                        for j in set(labels2))
        
        # Calcula índice
        sum_comb = sum(self._comb2(contingency_table[i][j]) 
                      for i in contingency_table for j in contingency_table[i])
        
        expected_index = sum_comb_c * sum_comb_k / self._comb2(n)
        max_index = (sum_comb_c + sum_comb_k) / 2
        
        if max_index - expected_index == 0:
            return 0.0
        
        return (sum_comb - expected_index) / (max_index - expected_index)
    
    def _comb2(self, n: int) -> int:
        """Calcula combinação C(n,2) = n*(n-1)/2"""
        return n * (n - 1) // 2 if n >= 2 else 0


In [5]:
class TemporalPatternAnalyzer:
    """
    Analisador especializado para reconhecimento de padrões temporais.
    
    Atributos:
        analyzer (TemporalNetworkAnalyzer): Referência ao analisador principal
        network_metrics (dict): Métricas da rede por período
        growth_patterns (dict): Padrões de crescimento identificados
        hub_evolution_patterns (dict): Padrões de evolução dos hubs
        competition_patterns (dict): Padrões competitivos/cooperativos
    """
    
    def __init__(self, analyzer: TemporalNetworkAnalyzer):
        """
        Inicializa o analisador de padrões temporais.
        
        Args:
            analyzer (TemporalNetworkAnalyzer): Instância do analisador principal
        """
        self.analyzer = analyzer
        self.network_metrics = {}
        self.growth_patterns = {}
        self.hub_evolution_patterns = {}
        self.competition_patterns = {}
    
    def calculate_network_metrics(self) -> Dict:
        """
        Calcula métricas básicas da rede para cada período temporal.
        
        Returns:
            Dict: Métricas da rede por período
        """
        if not self.analyzer.temporal_graphs:
            raise ValueError("Snapshots temporais não criados. Execute create_temporal_snapshots() primeiro.")
        
        print("Calculando métricas da rede ao longo do tempo...")
        
        for period, graph in self.analyzer.temporal_graphs.items():
            metrics = {}
            
            # Métricas básicas
            metrics['num_nodes'] = graph.number_of_nodes()
            metrics['num_edges'] = graph.number_of_edges()
            metrics['density'] = nx.density(graph)

            if graph.number_of_nodes() > 0:
                degrees = [d for n, d in graph.degree()]
                metrics['avg_degree'] = np.mean(degrees)
            else:
                metrics['avg_degree'] = 0.0
            
            # Métricas de conectividade
            if graph.number_of_edges() > 0:
                # Componentes conectados
                connected_components = list(nx.connected_components(graph))
                metrics['num_components'] = len(connected_components)
                
                # Tamanho do maior componente
                largest_cc = max(connected_components, key=len) if connected_components else set()
                metrics['largest_cc_size'] = len(largest_cc)
                metrics['largest_cc_ratio'] = len(largest_cc) / graph.number_of_nodes() if graph.number_of_nodes() > 0 else 0
                
                # Diâmetro (apenas para o maior componente)
                if len(largest_cc) > 1:
                    largest_cc_subgraph = graph.subgraph(largest_cc)
                    try:
                        metrics['diameter'] = nx.diameter(largest_cc_subgraph)
                    except:
                        metrics['diameter'] = -1
                else:
                    metrics['diameter'] = 0
                
                # Coeficiente de clustering
                try:
                    metrics['clustering_coefficient'] = nx.average_clustering(graph)
                except:
                    metrics['clustering_coefficient'] = 0
            else:
                # Grafo sem arestas
                metrics['num_components'] = graph.number_of_nodes()
                metrics['largest_cc_size'] = 1
                metrics['largest_cc_ratio'] = 1 / graph.number_of_nodes() if graph.number_of_nodes() > 0 else 0
                metrics['diameter'] = 0
                metrics['clustering_coefficient'] = 0
            
            self.network_metrics[period] = metrics
        
        print(f"✓ Métricas calculadas para {len(self.network_metrics)} períodos")
        
        return self.network_metrics
    
    def detect_growth_patterns(self) -> Dict:
        """
        Detecta padrões de crescimento na rede ao longo do tempo.
        
        Returns:
            Dict: Padrões de crescimento identificados
        """
        if not self.network_metrics:
            self.calculate_network_metrics()
        
        print("Detectando padrões de crescimento...")
        
        # Ordena períodos
        periods = sorted(self.network_metrics.keys())
        
        if len(periods) < 2:
            print("✗ Número insuficiente de períodos para análise de crescimento")
            return {}
        
        # Extrai séries temporais
        nodes_series = [self.network_metrics[p]['num_nodes'] for p in periods]
        edges_series = [self.network_metrics[p]['num_edges'] for p in periods]
        density_series = [self.network_metrics[p]['density'] for p in periods]
        
        # Calcula tendências
        nodes_trend = self._calculate_trend(nodes_series)
        edges_trend = self._calculate_trend(edges_series)
        density_trend = self._calculate_trend(density_series)
        
        # Calcula variação percentual
        nodes_change = (nodes_series[-1] - nodes_series[0]) / nodes_series[0] * 100 if nodes_series[0] > 0 else 0
        edges_change = (edges_series[-1] - edges_series[0]) / edges_series[0] * 100 if edges_series[0] > 0 else 0
        density_change = (density_series[-1] - density_series[0]) / density_series[0] * 100 if density_series[0] > 0 else 0
        
        # Identifica padrões
        self.growth_patterns = {
            'nodes_trend': nodes_trend,
            'edges_trend': edges_trend,
            'density_trend': density_trend,
            'nodes_change_percent': nodes_change,
            'edges_change_percent': edges_change,
            'density_change_percent': density_change,
            'nodes_series': nodes_series,
            'edges_series': edges_series,
            'density_series': density_series,
            'periods': periods
        }
        
        print(f"✓ Padrões de crescimento detectados:")
        print(f"  - Tendência de nós: {nodes_trend}")
        print(f"  - Tendência de arestas: {edges_trend}")
        print(f"  - Variação de nós: {nodes_change:.1f}%")
        print(f"  - Variação de arestas: {edges_change:.1f}%")
        
        return self.growth_patterns
    
    def detect_hub_evolution_patterns(self, hub_analyzer: 'HubAnalyzer') -> Dict:
        """Versão corrigida da função"""
        if not hub_analyzer.hub_history:
            raise ValueError("Execute hub_analyzer.analyze_temporal_hubs() primeiro.")
        
        print("Detectando padrões de evolução dos hubs...")
        
        # CORREÇÃO: Usar hub_persistence corretamente
        persistent_hubs = [hub for hub, fraction in hub_analyzer.hub_persistence.items() 
                        if fraction >= 0.3]
        
        # Rastreia evolução da centralidade dos hubs persistentes
        hub_evolution = {}
        
        for hub in persistent_hubs:
            evolution = []
            for period in sorted(hub_analyzer.centrality_history.keys()):
                centrality = hub_analyzer.centrality_history[period].get(
                    hub_analyzer.method_used, {}).get(hub, 0)
                evolution.append(centrality)
            
            # Detecta padrão de evolução
            if len(evolution) >= 2:
                trend = self._calculate_trend(evolution)
                volatility = np.std(evolution) / np.mean(evolution) if np.mean(evolution) > 0 else 0
                
                hub_evolution[hub] = {
                    'trend': trend,
                    'volatility': volatility,
                    'evolution': evolution
                }
        
        # Classifica hubs por padrão
        growing_hubs = [hub for hub, data in hub_evolution.items() 
                    if data['trend'] == 'crescente']
        declining_hubs = [hub for hub, data in hub_evolution.items() 
                        if data['trend'] == 'decrescente']
        volatile_hubs = [hub for hub, data in hub_evolution.items() 
                    if data['volatility'] > 0.5]
        
        self.hub_evolution_patterns = {
            'hub_evolution': hub_evolution,
            'growing_hubs': growing_hubs,
            'declining_hubs': declining_hubs,
            'volatile_hubs': volatile_hubs
        }
        
        print(f"✓ Padrões de evolução dos hubs detectados:")
        print(f"  - Hubs crescentes: {len(growing_hubs)}")
        print(f"  - Hubs decrescentes: {len(declining_hubs)}")
        print(f"  - Hubs voláteis: {len(volatile_hubs)}")
        
        return self.hub_evolution_patterns
    
    def analyze_competition_vs_cooperation(self, hub_analyzer: 'HubAnalyzer') -> Dict:
        """
        Analisa dinâmicas competitivas vs cooperativas entre hubs.
        
        Args:
            hub_analyzer (HubAnalyzer): Analisador de hubs com histórico
            
        Returns:
            Dict: Padrões competitivos/cooperativos identificados
        """
        if not hub_analyzer.hub_history:
            raise ValueError("Execute hub_analyzer.analyze_temporal_hubs() primeiro.")
        
        if not hasattr(hub_analyzer, 'hub_persistence') or not hub_analyzer.hub_persistence:
            print("Executando análise de hubs para obter persistência...")
            hub_analyzer.analyze_temporal_hubs()

        print("Analisando dinâmicas competitivas vs cooperativas...")
        
        # Identifica hubs persistentes
        persistent_hubs = [hub for hub, persistence in hub_analyzer.hub_persistence.items() 
                         if persistence >= 0.3]
        
        if len(persistent_hubs) < 2:
            print("✗ Número insuficiente de hubs persistentes para análise")
            return {}
        
        # Calcula correlações entre centralidades de hubs
        correlations = {}
        
        for i, hub1 in enumerate(persistent_hubs):
            for hub2 in persistent_hubs[i+1:]:
                # Extrai séries temporais de centralidade
                hub1_series = []
                hub2_series = []
                
                for period in sorted(hub_analyzer.centrality_history.keys()):
                    centrality_dict = hub_analyzer.centrality_history[period].get(
                        hub_analyzer.method_used, {})
                    
                    hub1_centrality = centrality_dict.get(hub1, 0)
                    hub2_centrality = centrality_dict.get(hub2, 0)
                    
                    hub1_series.append(hub1_centrality)
                    hub2_series.append(hub2_centrality)
                
                # Calcula correlação
                if len(hub1_series) >= 3 and np.std(hub1_series) > 0 and np.std(hub2_series) > 0:
                    correlation = np.corrcoef(hub1_series, hub2_series)[0, 1]
                    correlations[(hub1, hub2)] = correlation
        
        # Classifica relações
        cooperative_pairs = [(h1, h2) for (h1, h2), corr in correlations.items() 
                           if corr >= 0.5]
        competitive_pairs = [(h1, h2) for (h1, h2), corr in correlations.items() 
                           if corr <= -0.5]
        neutral_pairs = [(h1, h2) for (h1, h2), corr in correlations.items() 
                       if -0.5 < corr < 0.5]
        
        self.competition_patterns = {
            'correlations': correlations,
            'cooperative_pairs': cooperative_pairs,
            'competitive_pairs': competitive_pairs,
            'neutral_pairs': neutral_pairs
        }
        
        print(f"✓ Dinâmicas identificadas:")
        print(f"  - Pares cooperativos: {len(cooperative_pairs)}")
        print(f"  - Pares competitivos: {len(competitive_pairs)}")
        print(f"  - Pares neutros: {len(neutral_pairs)}")
        
        # Determina padrão dominante
        if len(cooperative_pairs) > len(competitive_pairs):
            dominant_pattern = "cooperativo"
        elif len(competitive_pairs) > len(cooperative_pairs):
            dominant_pattern = "competitivo"
        else:
            dominant_pattern = "neutro"
        
        print(f"  - Padrão dominante: {dominant_pattern}")
        self.competition_patterns['dominant_pattern'] = dominant_pattern
        
        return self.competition_patterns
    
    def _calculate_trend(self, series: List[float]) -> str:
        """
        Calcula a tendência de uma série temporal.
        
        Args:
            series (List[float]): Série temporal
            
        Returns:
            str: Tendência ('crescente', 'decrescente', 'estável')
        """
        if len(series) < 2:
            return 'estável'
        
        # Calcula coeficiente angular da regressão linear
        x = np.arange(len(series))
        y = np.array(series)
        
        if np.std(y) == 0:
            return 'estável'
        
        slope, _ = np.polyfit(x, y, 1)
        
        # Determina tendência com base no coeficiente angular
        if slope > 0.05 * np.mean(y) / len(y):
            return 'crescente'
        elif slope < -0.05 * np.mean(y) / len(y):
            return 'decrescente'
        else:
            return 'estável'


In [6]:
class TemporalNetworkVisualizer:
    """
    Visualizador especializado para redes temporais.
    
    Atributos:
        analyzer (TemporalNetworkAnalyzer): Referência ao analisador principal
    """
    
    def __init__(self, analyzer: TemporalNetworkAnalyzer):
        """
        Inicializa o visualizador de redes temporais.
        
        Args:
            analyzer (TemporalNetworkAnalyzer): Instância do analisador principal
        """
        self.analyzer = analyzer
    
    def plot_network_evolution(self, metrics: List[str] = ['num_nodes', 'num_edges', 'density', 'avg_degree']) -> pd.DataFrame:
        """
        Prepara dados para visualização da evolução da rede.
        
        Args:
            metrics (List[str]): Lista de métricas para visualizar
            
        Returns:
            pd.DataFrame: DataFrame com dados para visualização
        """
        if not hasattr(self.analyzer, 'temporal_graphs') or not self.analyzer.temporal_graphs:
            raise ValueError("Snapshots temporais não criados. Execute create_temporal_snapshots() primeiro.")
        
        print("Preparando dados para visualização da evolução da rede...")
        
        # Cria DataFrame com métricas básicas
        data = []
        
        for period, graph in self.analyzer.temporal_graphs.items():
            row = {'period': period}
            
            # Métricas básicas
            if 'num_nodes' in metrics:
                row['num_nodes'] = graph.number_of_nodes()
            
            if 'num_edges' in metrics:
                row['num_edges'] = graph.number_of_edges()
            
            if 'density' in metrics:
                row['density'] = nx.density(graph)
            
            if 'avg_degree' in metrics:
                if graph.number_of_nodes() > 0:
                    row['avg_degree'] = 2 * graph.number_of_edges() / graph.number_of_nodes()
                else:
                    row['avg_degree'] = 0
            
            if 'num_components' in metrics:
                row['num_components'] = nx.number_connected_components(graph)
            
            data.append(row)
        
        # Cria DataFrame
        df = pd.DataFrame(data)
        
        # Salva dados para visualização externa
        df.to_csv('network_evolution_data.csv', index=False)
        
        print(f"✓ Dados preparados para {len(df)} períodos")
        print(f"✓ Arquivo salvo: network_evolution_data.csv")
        
        return df
    
    def prepare_hub_evolution_data(self, hub_analyzer: 'HubAnalyzer', 
                                 top_k: int = 5) -> pd.DataFrame:
        """
        Prepara dados para visualização da evolução dos hubs.
        
        Args:
            hub_analyzer (HubAnalyzer): Analisador de hubs
            top_k (int): Número de hubs para visualizar
            
        Returns:
            pd.DataFrame: DataFrame com dados para visualização
        """
        if not hub_analyzer.hub_history:
            raise ValueError("Execute hub_analyzer.analyze_temporal_hubs() primeiro.")
        
        print(f"Preparando dados para visualização dos top-{top_k} hubs...")
        
        # Identifica os top-k hubs mais persistentes
        persistent_hubs = sorted(hub_analyzer.hub_persistence.items(), 
                               key=lambda x: x[1], reverse=True)[:top_k]
        
        # Cria DataFrame com evolução dos hubs
        data = []
        
        for period in sorted(hub_analyzer.centrality_history.keys()):
            row = {'period': period}
            
            for hub, _ in persistent_hubs:
                centrality = hub_analyzer.centrality_history[period].get(
                    hub_analyzer.method_used, {}).get(hub, 0)
                row[f'hub_{hub}'] = centrality
            
            data.append(row)
        
        # Cria DataFrame
        df = pd.DataFrame(data)
        
        # Salva dados para visualização externa
        df.to_csv('hub_evolution_data.csv', index=False)
        
        print(f"✓ Dados preparados para {len(df)} períodos")
        print(f"✓ Arquivo salvo: hub_evolution_data.csv")
        
        return df
    
    def prepare_community_evolution_data(self, cluster_analyzer: 'TemporalClusterAnalyzer') -> pd.DataFrame:
        """
        Prepara dados para visualização da evolução das comunidades.
        
        Args:
            cluster_analyzer (TemporalClusterAnalyzer): Analisador de comunidades
            
        Returns:
            pd.DataFrame: DataFrame com dados para visualização
        """
        if not cluster_analyzer.community_history:
            raise ValueError("Execute cluster_analyzer.analyze_temporal_communities() primeiro.")
        
        print("Preparando dados para visualização da evolução das comunidades...")
        
        # Cria DataFrame com evolução das comunidades
        data = []
        
        for period in sorted(cluster_analyzer.community_history.keys()):
            communities = cluster_analyzer.community_history[period]
            modularity = cluster_analyzer.modularity_history.get(period, 0)
            
            # Conta número de comunidades
            num_communities = len(set(communities.values()))
            
            # Calcula tamanho médio das comunidades
            community_sizes = Counter(communities.values())
            avg_size = np.mean(list(community_sizes.values())) if community_sizes else 0
            
            # Calcula tamanho da maior comunidade
            largest_community_size = max(community_sizes.values()) if community_sizes else 0
            
            row = {
                'period': period,
                'num_communities': num_communities,
                'avg_community_size': avg_size,
                'largest_community_size': largest_community_size,
                'modularity': modularity
            }
            
            data.append(row)
        
        # Cria DataFrame
        df = pd.DataFrame(data)
        
        # Salva dados para visualização externa
        df.to_csv('community_evolution_data.csv', index=False)
        
        print(f"✓ Dados preparados para {len(df)} períodos")
        print(f"✓ Arquivo salvo: community_evolution_data.csv")
        
        return df
    
    def generate_summary_report(self, hub_analyzer: 'HubAnalyzer', 
                              cluster_analyzer: 'TemporalClusterAnalyzer',
                              pattern_analyzer: 'TemporalPatternAnalyzer',
                              metrics: List[str] = ['num_nodes', 'num_edges', 'density', 'avg_degree'],
                              ) -> Dict:
        """
        Gera um relatório executivo completo da análise.
        
        Args:
            hub_analyzer (HubAnalyzer): Analisador de hubs
            cluster_analyzer (TemporalClusterAnalyzer): Analisador de comunidades
            pattern_analyzer (TemporalPatternAnalyzer): Analisador de padrões
            
        Returns:
            Dict: Relatório executivo completo
        """
        print("Gerando relatório executivo...")
        
        # Verifica se todas as análises foram executadas
        if not hub_analyzer.hub_history:
            print("Executando análise de hubs...")
            hub_analyzer.analyze_temporal_hubs()
        
        if not cluster_analyzer.community_history:
            print("Executando análise de comunidades...")
            cluster_analyzer.analyze_temporal_communities()
        
        if not pattern_analyzer.network_metrics:
            print("Executando análise de métricas da rede...")
            pattern_analyzer.calculate_network_metrics()
        
        if not pattern_analyzer.growth_patterns:
            print("Executando análise de padrões de crescimento...")
            pattern_analyzer.detect_growth_patterns()
        
        # Prepara dados para visualização
        self.plot_network_evolution(metrics)
        self.prepare_hub_evolution_data(hub_analyzer)
        self.prepare_community_evolution_data(cluster_analyzer)
        
        # Compila relatório executivo
        report = {
            'dataset_info': {
                'num_nodes': len(self.analyzer.nodes),
                'num_edges': len(self.analyzer.edges),
                'time_window': self.analyzer.time_window,
                'num_periods': len(self.analyzer.temporal_graphs)
            },
            'hub_analysis': {
                'total_hubs': hub_analyzer.hub_statistics()['total_unique_hubs'],
                'persistent_hubs': hub_analyzer.hub_statistics()['highly_persistent_hubs'] + 
                                  hub_analyzer.hub_statistics()['moderately_persistent_hubs'],
                'top_hubs': [hub for hub, _ in sorted(hub_analyzer.hub_persistence.items(), 
                                                   key=lambda x: x[1], reverse=True)[:5]]
            },
            'community_analysis': {
                'avg_num_communities': np.mean(cluster_analyzer.num_communities_per_period),
                'avg_community_size': np.mean(cluster_analyzer.avg_community_size_per_period),
                'temporal_stability': cluster_analyzer.avg_stability,
                'avg_modularity': np.mean(list(cluster_analyzer.modularity_history.values()))
            },
            'growth_patterns': {
                'nodes_trend': pattern_analyzer.growth_patterns.get('nodes_trend', 'N/A'),
                'edges_trend': pattern_analyzer.growth_patterns.get('edges_trend', 'N/A'),
                'nodes_change': pattern_analyzer.growth_patterns.get('nodes_change_percent', 0),
                'edges_change': pattern_analyzer.growth_patterns.get('edges_change_percent', 0)
            }
        }
        
        # Adiciona análise de competição vs cooperação se disponível
        if hasattr(pattern_analyzer, 'competition_patterns') and pattern_analyzer.competition_patterns:
            report['competition_analysis'] = {
                'dominant_pattern': pattern_analyzer.competition_patterns.get('dominant_pattern', 'N/A'),
                'cooperative_pairs': len(pattern_analyzer.competition_patterns.get('cooperative_pairs', [])),
                'competitive_pairs': len(pattern_analyzer.competition_patterns.get('competitive_pairs', []))
            }
        
        print("✓ Relatório executivo gerado com sucesso!")
        
        return report


In [7]:
def generate_synthetic_data(num_nodes: int = 100, 
                          num_periods: int = 10, 
                          avg_edges_per_period: int = 200,
                          seed: int = 42) -> Tuple[pd.DataFrame, TemporalNetworkAnalyzer]:
    """
    Gera dados sintéticos para teste do framework.
    
    Args:
        num_nodes (int): Número de nós na rede
        num_periods (int): Número de períodos temporais
        avg_edges_per_period (int): Média de arestas por período
        seed (int): Seed para reprodutibilidade
        
    Returns:
        Tuple[pd.DataFrame, TemporalNetworkAnalyzer]: DataFrame com arestas e analisador configurado
    """
    np.random.seed(seed)
    print(f"Gerando dados sintéticos com {num_nodes} nós e {num_periods} períodos...")
    
    # Cria lista de arestas
    data = []
    
    # Timestamp base (1 de janeiro de 2020)
    base_timestamp = 1577836800
    
    for period in range(num_periods):
        # Número de arestas para este período (variação aleatória)
        num_edges = int(np.random.normal(avg_edges_per_period, avg_edges_per_period * 0.2))
        num_edges = max(1, num_edges)
        
        # Timestamp para este período (1 dia por período)
        period_timestamp = base_timestamp + period * 86400
        
        # Gera arestas aleatórias
        for _ in range(num_edges):
            source = np.random.randint(0, num_nodes)
            target = np.random.randint(0, num_nodes)
            
            # Evita self-loops
            while target == source:
                target = np.random.randint(0, num_nodes)
            
            # Adiciona aresta com timestamp
            data.append([source, target, period_timestamp])
    
    # Cria DataFrame
    edges_df = pd.DataFrame(data, columns=['source', 'target', 'timestamp'])
    
    # Configura analisador
    analyzer = TemporalNetworkAnalyzer()
    analyzer.edges = edges_df.copy()
    analyzer.edges['datetime'] = pd.to_datetime(analyzer.edges['timestamp'], unit='s')
    analyzer.nodes = set(range(num_nodes))
    
    # Cria bins temporais
    start_time = analyzer.edges['datetime'].min()
    end_time = analyzer.edges['datetime'].max()
    analyzer.time_window = 'D'
    analyzer.time_bins = pd.date_range(start=start_time, end=end_time, freq='D')
    
    # Atribui cada aresta a um bin temporal
    analyzer.edges['time_bin'] = pd.cut(analyzer.edges['datetime'], 
                                      bins=analyzer.time_bins, 
                                      labels=False, 
                                      include_lowest=True)
    
    # Cria snapshots
    analyzer.temporal_graphs = {}
    
    for bin_idx in range(len(analyzer.time_bins) - 1):
        # Filtra arestas do período atual
        period_edges = analyzer.edges[analyzer.edges['time_bin'] == bin_idx]
        
        # Cria grafo para o período
        G = nx.Graph()
        
        # Adiciona nós (todos os nós que existem no dataset)
        G.add_nodes_from(analyzer.nodes)
        
        # Adiciona arestas do período
        if not period_edges.empty:
            edge_list = list(zip(period_edges['source'], period_edges['target']))
            G.add_edges_from(edge_list)
        
        analyzer.temporal_graphs[bin_idx] = G
    
    print(f"✓ Dados sintéticos gerados:")
    print(f"  - {len(edges_df)} arestas temporais")
    print(f"  - {num_nodes} nós")
    print(f"  - {len(analyzer.temporal_graphs)} snapshots")
    
    return edges_df, analyzer


In [8]:
def run_basic_example():
    """
    Executa um exemplo básico do framework com dados sintéticos.
    """
    print("=== EXEMPLO BÁSICO DE USO DO FRAMEWORK ===")
    
    # 1. Gera dados sintéticos
    print("\n1. GERAÇÃO DE DADOS SINTÉTICOS")
    _, analyzer = generate_synthetic_data(num_nodes=50, num_periods=5, avg_edges_per_period=100)
    
    # 2. Inicializa analisadores especializados
    print("\n2. INICIALIZAÇÃO DOS ANALISADORES")
    hub_analyzer = HubAnalyzer(analyzer)
    cluster_analyzer = TemporalClusterAnalyzer(analyzer)
    pattern_analyzer = TemporalPatternAnalyzer(analyzer)
    
    # 3. Análise de hubs
    print("\n3. ANÁLISE DE HUBS")
    for period, graph in analyzer.temporal_graphs.items():
        print(f"\nPeríodo {period}:")
        hubs = hub_analyzer.identify_hubs(graph, top_k=3, method='degree')
        print(f"Top 3 hubs: {hubs}")
    
    # 4. Análise de comunidades
    print("\n4. ANÁLISE DE COMUNIDADES")
    for period, graph in analyzer.temporal_graphs.items():
        print(f"\nPeríodo {period}:")
        communities, modularity = cluster_analyzer.detect_communities(graph)
        num_communities = len(set(communities.values()))
        print(f"Número de comunidades: {num_communities}")
        print(f"Modularidade: {modularity:.3f}")
    
    # 5. Análise de métricas da rede
    print("\n5. ANÁLISE DE MÉTRICAS DA REDE")
    metrics = pattern_analyzer.calculate_network_metrics()
    
    print("\nMétricas por período:")
    for period, period_metrics in metrics.items():
        print(f"\nPeríodo {period}:")
        for metric, value in period_metrics.items():
            print(f"  - {metric}: {value}")
    
    print("\n=== EXEMPLO BÁSICO CONCLUÍDO COM SUCESSO ===")

    
    
    return metrics


In [9]:
# Inicializar o analisador principal
analyzer = TemporalNetworkAnalyzer()

# Baixar e carregar o dataset
analyzer.download_dataset()
analyzer.load_dataset("CollegeMsg.txt.gz")

# Preprocessar dados com janela temporal diária
analyzer.preprocess_data(time_window='D')

# Criar snapshots temporais
analyzer.create_temporal_snapshots()

# Inicializar analisadores especializados
hub_analyzer = HubAnalyzer(analyzer)
cluster_analyzer = TemporalClusterAnalyzer(analyzer)
pattern_analyzer = TemporalPatternAnalyzer(analyzer)
visualizer = TemporalNetworkVisualizer(analyzer)

# Executar análise completa
hub_results = hub_analyzer.analyze_temporal_hubs(top_k=10, method='degree')
hub_stats = hub_analyzer.hub_statistics()

community_results = cluster_analyzer.analyze_temporal_communities(method='louvain')

pattern_analyzer.calculate_network_metrics()
pattern_analyzer.detect_growth_patterns()
pattern_analyzer.analyze_competition_vs_cooperation(hub_analyzer)
pattern_analyzer.detect_hub_evolution_patterns(hub_analyzer)

# Visualizar resultados
network_evolution = visualizer.plot_network_evolution(metrics=['num_nodes', 'num_edges', 'density', 'avg_degree'])
hub_evolution = visualizer.prepare_hub_evolution_data(hub_analyzer, top_k=5)
community_evolution = visualizer.prepare_community_evolution_data(cluster_analyzer)

# Gerar relatório final
report = visualizer.generate_summary_report(hub_analyzer, cluster_analyzer, pattern_analyzer)

Baixando dataset CollegeMsg...
✓ Dataset baixado: CollegeMsg.txt.gz
Carregando dataset...
✓ Dataset carregado:
  - 59835 arestas temporais
  - 1899 nós únicos
  - Período: 2004-04-15 14:56:01 a 2004-10-26 07:52:22
Preprocessando dados com janela temporal: D
✓ Dados preprocessados:
  - 193 períodos temporais
  - 59795 arestas válidas
Criando snapshots temporais...
✓ 193 snapshots criados
  - Média de nós por snapshot: 1899.0
  - Média de arestas por snapshot: 133.8
Analisando evolução temporal dos hubs (método: degree, top-10)...
✓ Análise concluída:
  - 436 nós identificados como hubs
  - Top 3 hubs mais persistentes:
    1. Nó 9: 31.1% do tempo
    2. Nó 95: 26.9% do tempo
    3. Nó 32: 25.4% do tempo
Analisando evolução temporal das comunidades (método: louvain)...
✓ Análise concluída:
  - Média de comunidades: 1800.9
  - Tamanho médio: 1.1
  - Estabilidade: 0.100
  - Modularidade: 0.762
Calculando métricas da rede ao longo do tempo...
✓ Métricas calculadas para 193 períodos
Detectan

In [10]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def plot_interactive_evolution():
    """Visualização interativa corrigida para análise de redes temporais"""
    try:
        # Carregar dados
        network_df = pd.read_csv('network_evolution_data.csv')
        hubs_df = pd.read_csv('hub_evolution_data.csv')
        communities_df = pd.read_csv('community_evolution_data.csv')
    except FileNotFoundError:
        print("Arquivos não encontrados. Execute primeiro as etapas de análise.")
        return

    # Criar figura com subplots
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            'Evolução da Rede', 
            'Evolução dos Hubs',
            'Evolução das Comunidades',
            'Métricas de Densidade'
        ),
        specs=[[{"type": "scatter"}, {"type": "scatter"}],
               [{"type": "scatter"}, {"type": "scatter"}]],
        vertical_spacing=0.15,
        horizontal_spacing=0.1
    )

    # Gráfico 1: Evolução da rede
    fig.add_trace(go.Scatter(
        x=network_df['period'], 
        y=network_df['num_nodes'],
        name='Nós',
        line=dict(color='royalblue', width=3),
        mode='lines+markers'
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=network_df['period'], 
        y=network_df['num_edges'],
        name='Arestas',
        line=dict(color='firebrick', width=3),
        mode='lines+markers'
    ), row=1, col=1)

    # Gráfico 2: Evolução dos hubs
    for col in hubs_df.columns:
        if col != 'period' and 'hub_' in col:
            fig.add_trace(go.Scatter(
                x=hubs_df['period'], 
                y=hubs_df[col],
                name=f'Hub {col.split("_")[1]}',
                mode='lines+markers',
                showlegend=True
            ), row=1, col=2)

    # Gráfico 3: Evolução das comunidades
    fig.add_trace(go.Scatter(
        x=communities_df['period'], 
        y=communities_df['num_communities'],
        name='Nº Comunidades',
        line=dict(color='green', width=3),
        mode='lines+markers'
    ), row=2, col=1)

    fig.add_trace(go.Scatter(
        x=communities_df['period'], 
        y=communities_df['avg_community_size'],
        name='Tamanho Médio',
        line=dict(color='purple', width=3),
        mode='lines+markers'
    ), row=2, col=1)

    # Gráfico 4: Densidade e grau médio
    fig.add_trace(go.Scatter(
        x=network_df['period'], 
        y=network_df['density'],
        name='Densidade',
        line=dict(color='orange', width=3),
        mode='lines+markers'
    ), row=2, col=2)

    fig.add_trace(go.Scatter(
        x=network_df['period'], 
        y=network_df['avg_degree'],
        name='Grau Médio',
        line=dict(color='cyan', width=3),
        mode='lines+markers'
    ), row=2, col=2)

    # Atualizar layout
    fig.update_layout(
        title_text='Análise Completa de Rede Temporal',
        height=900,
        width=1200,
        template='plotly_dark',
        hovermode='x unified',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    # Atualizar eixos
    fig.update_xaxes(title_text='Período', row=1, col=1)
    fig.update_yaxes(title_text='Quantidade', row=1, col=1)
    fig.update_xaxes(title_text='Período', row=1, col=2)
    fig.update_yaxes(title_text='Centralidade', row=1, col=2)
    fig.update_xaxes(title_text='Período', row=2, col=1)
    fig.update_yaxes(title_text='Valor', row=2, col=1)
    fig.update_xaxes(title_text='Período', row=2, col=2)
    fig.update_yaxes(title_text='Valor', row=2, col=2)

    fig.show()
    return fig

_ = plot_interactive_evolution()